# Cross-Model Crystallization: "Transformers Know Before They Speak"
#
# **Goal**: Replicate the crystallization findings from Qwen2.5-3B on **Qwen2.5-7B** to test universality.
# Same architecture family, 2.3x more parameters. Isolates SCALE as the only variable.
#
# ## Key measurements:
# 1. **Easy math crystallization** — does p(answer) appear at t=0 before any generation?
# 2. **Difficulty gradient** — does crystallization token shift right with problem difficulty?
# 3. **Non-math tasks** — does "knows before speaks" hold for factual recall, translation?
# 4. **Constrained decoding** — if we force the answer at t=0, does accuracy match CoT?
# 5. **Language flip replication** — does the MLP flip effect generalize to 7B?
# 6. **Participation ratio** — does effective dimensionality narrow through the layer stack?
#
# ## What we found on 3B (36 layers, d=2048):
# - Chinese baseline at t=0, L35: p(answer) = 29%. Model already partially knows.
# - English baseline at t=0, L35: p(answer) = 2.6%. 11x lower.
# - Difficulty gradient: easy=t4.5, medium=t7.3, hard=t7.3, AIME=t15.5
# - Flip makes crystallization instantaneous (mean token 4.0 → 0.0)

In [ ]:
# Cell 1: Setup
!pip install -q transformers accelerate torch

import json
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
import time
from pathlib import Path

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Load model — Qwen2.5-7B (same arch as 3B, 2.3x scale)
# 28 layers, d=3584. ~14GB bf16, fits on H100/A100 easily.
MODEL_NAME = "Qwen/Qwen2.5-7B"
SEED = 42

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cuda"
)
model.eval()

n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"Loaded in {time.time()-t0:.1f}s: {n_layers} layers, d={d_model}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Quick sanity check
inp = tokenizer("Calculate 47 + 86.", return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=30)
print(f"Sanity: {tokenizer.decode(out[0], skip_special_tokens=True)[:100]}")

In [ ]:
# Cell 3: Core crystallization engine
# Measures p(answer_token) at every (layer, generated_token) position

ALL_LAYERS = list(range(n_layers))

def get_answer_token_ids(tokenizer, answer):
    """Get all token IDs that could represent the answer."""
    ans_str = str(answer)
    ids = set()
    for prefix in ["", " ", "\n"]:
        toks = tokenizer.encode(prefix + ans_str, add_special_tokens=False)
        ids.update(toks)
    toks = tokenizer.encode(ans_str, add_special_tokens=False)
    ids.update(toks)
    return list(ids)


def generate_with_crystallization(model, tokenizer, prompt, answer,
                                  flip_dirs=None, flip_scale=-1.0,
                                  flip_layers=None, max_new=80):
    """
    Generate token by token. At each step:
    1. Run forward pass, capturing hidden states at every layer
    2. Early-exit: apply final layernorm + lm_head to each layer's hidden state
    3. Record p(answer_token) at each (layer, token)

    Optional: apply MLP delta flip intervention if flip_dirs provided.

    Returns: grid (n_tokens x n_layers), gen_text, gen_token_ids
    """
    device = next(model.parameters()).device
    answer_token_ids = get_answer_token_ids(tokenizer, answer)

    # Access model internals — Qwen3.5 uses same architecture as Qwen2.5
    final_ln = model.model.norm
    lm_head = model.lm_head

    input_ids = tokenizer.encode(prompt, add_special_tokens=True)
    input_ids = torch.tensor([input_ids], device=device)

    crystallization = []
    generated_tokens = []

    # Set up flip hooks if provided
    flip_handles = []
    if flip_dirs is not None and flip_layers is not None:
        def make_flip(l):
            v = flip_dirs[l].to(device)
            def hook(module, inp, out):
                h = out
                proj = torch.einsum("...d,d->...", h, v)
                return h + flip_scale * proj.unsqueeze(-1) * v
            return hook
        for l in flip_layers:
            if l in flip_dirs:
                handle = model.model.layers[l].mlp.register_forward_hook(make_flip(l))
                flip_handles.append(handle)

    try:
        for t in range(max_new):
            layer_hiddens = {}

            def make_capture(l):
                def hook(module, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    layer_hiddens[l] = h[:, -1:, :].detach()
                return hook

            cap_handles = [model.model.layers[l].register_forward_hook(make_capture(l))
                           for l in ALL_LAYERS]

            with torch.no_grad():
                outputs = model(input_ids)

            for h in cap_handles:
                h.remove()

            next_logits = outputs.logits[:, -1, :]
            next_token = next_logits.argmax(dim=-1)

            # Early-exit probabilities at each layer
            layer_probs = {}
            for l in ALL_LAYERS:
                h_l = layer_hiddens[l]
                h_normed = final_ln(h_l)
                logits_l = lm_head(h_normed).float().squeeze(0).squeeze(0)
                probs = F.softmax(logits_l, dim=-1)
                p_answer = max(probs[tid].item() for tid in answer_token_ids) if answer_token_ids else 0.0
                layer_probs[l] = p_answer

            crystallization.append(layer_probs)
            generated_tokens.append(next_token.item())

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    finally:
        for h in flip_handles:
            h.remove()

    gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    n_tok = len(crystallization)
    grid = np.zeros((n_tok, n_layers))
    for t_idx in range(n_tok):
        for l in ALL_LAYERS:
            grid[t_idx, l] = crystallization[t_idx].get(l, 0.0)

    return grid, gen_text, generated_tokens

In [ ]:
# Cell 4: Fit language direction (ZH - EN mean difference per MLP layer)

def fit_language_dirs(model, tokenizer, n_layers, strip_layers):
    """Fit 1D language direction per layer from ZH/EN math prompts."""
    rng = random.Random(SEED)
    problems = []
    for _ in range(50):
        a, b = rng.randint(10, 999), rng.randint(10, 999)
        problems.append({"zh": f"计算 {a} + {b} 的值。", "en": f"Calculate {a} + {b}."})
    for _ in range(50):
        n_val = rng.randint(5, 20); k_val = rng.randint(1, min(n_val-1, 8))
        problems.append({"zh": f"求组合数 C({n_val}, {k_val}) 的值。",
                         "en": f"Find the value of C({n_val}, {k_val})."})
    for _ in range(50):
        a = rng.randint(50, 9999); b_val = rng.randint(3, 37)
        problems.append({"zh": f"{a} 除以 {b_val} 的余数是多少？",
                         "en": f"What is the remainder when {a} is divided by {b_val}?"})
    for _ in range(50):
        w, h = rng.randint(2, 50), rng.randint(2, 50)
        problems.append({"zh": f"一个长方形的长为 {w}，宽为 {h}，求其面积。",
                         "en": f"A rectangle has length {w} and width {h}. Find its area."})
    rng.shuffle(problems)

    device = next(model.parameters()).device
    layer_acts = {l: {"zh": [], "en": []} for l in strip_layers}
    layer_out = {}

    def make_hook(l):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            layer_out[l] = h.detach().cpu().squeeze(0)[-1].float().numpy()
        return hook

    handles = [model.model.layers[l].register_forward_hook(make_hook(l)) for l in strip_layers]
    try:
        for lang in ["zh", "en"]:
            for p in problems:
                inp = tokenizer(p[lang], return_tensors="pt").to(device)
                with torch.no_grad():
                    model(**inp)
                for l in strip_layers:
                    layer_acts[l][lang].append(layer_out[l].copy())
                layer_out.clear()
    finally:
        for h in handles:
            h.remove()

    dirs = {}
    for l in strip_layers:
        zh_m = np.mean(layer_acts[l]["zh"], axis=0)
        en_m = np.mean(layer_acts[l]["en"], axis=0)
        v = zh_m - en_m
        dirs[l] = torch.tensor(v / (np.linalg.norm(v) + 1e-8), dtype=torch.bfloat16)
    return dirs


# Determine strip layers: middle ~50% of model
# For 3B (36 layers): L9-L26. For 9B, scale proportionally.
strip_start = n_layers // 4          # ~25%
strip_end = 3 * n_layers // 4 + 2    # ~75%+2
STRIP_LAYERS = list(range(strip_start, strip_end))
print(f"Strip layers: L{strip_start}-L{strip_end-1} ({len(STRIP_LAYERS)} layers)")

print("Fitting language directions (200 problems × 2 languages)...")
t0 = time.time()
lang_dirs = fit_language_dirs(model, tokenizer, n_layers, STRIP_LAYERS)
print(f"Done in {time.time()-t0:.1f}s")

In [ ]:
# Cell 5: SECTION A — Math Crystallization (Easy)
# Direct comparison with 3B results

MATH_EASY = [
    {"en": "Calculate 47 + 86.", "zh": "计算 47 + 86 的值。", "answer": 133},
    {"en": "Calculate 15 × 8.", "zh": "计算 15 × 8 的值。", "answer": 120},
    {"en": "Find the value of C(10, 3).", "zh": "求组合数 C(10, 3) 的值。", "answer": 120},
    {"en": "What is the remainder when 100 is divided by 7?",
     "zh": "100 除以 7 的余数是多少？", "answer": 2},
    {"en": "A rectangle has length 12 and width 5. Find its area.",
     "zh": "一个长方形的长为 12，宽为 5，求其面积。", "answer": 60},
    {"en": "Calculate 256 + 789.", "zh": "计算 256 + 789 的值。", "answer": 1045},
    {"en": "Find the value of C(8, 2).", "zh": "求组合数 C(8, 2) 的值。", "answer": 28},
    {"en": "What is the remainder when 500 is divided by 13?",
     "zh": "500 除以 13 的余数是多少？", "answer": 6},
    {"en": "Calculate 64 × 15.", "zh": "计算 64 × 15 的值。", "answer": 960},
    {"en": "A rectangle has length 30 and width 18. Find its area.",
     "zh": "一个长方形的长为 30，宽为 18，求其面积。", "answer": 540},
]

conditions = {
    "baseline_zh": {"lang": "zh", "flip": False},
    "flip_zh":     {"lang": "zh", "flip": True},
    "baseline_en": {"lang": "en", "flip": False},
}

math_easy_results = {}
for cond_name, cond in conditions.items():
    print(f"\n=== {cond_name} ===")
    results = []
    for pi, prob in enumerate(MATH_EASY):
        prompt = prob[cond["lang"]]
        grid, gen_text, gen_toks = generate_with_crystallization(
            model, tokenizer, prompt, prob["answer"],
            flip_dirs=lang_dirs if cond["flip"] else None,
            flip_layers=STRIP_LAYERS if cond["flip"] else None,
            max_new=80
        )

        # t=0 profile
        t0_L_last = float(grid[0, n_layers-1]) if grid.shape[0] > 0 else 0.0

        # Crystallization: first token where p > 0.1 at final layer
        cryst_tok = -1
        for t_idx in range(grid.shape[0]):
            if grid[t_idx, n_layers-1] > 0.1:
                cryst_tok = t_idx
                break

        ans_str = str(prob["answer"])
        answer_found = ans_str in gen_text

        results.append({
            "pi": pi, "answer": prob["answer"],
            "t0_p_final_layer": round(t0_L_last, 6),
            "crystallization_token": cryst_tok,
            "answer_found": answer_found,
            "n_tokens": grid.shape[0],
            "gen_text": gen_text[:200],
            "layer_profile_t0": [round(float(grid[0, l]), 6) for l in range(n_layers)]
                if grid.shape[0] > 0 else [],
            "token_profile_final": [round(float(grid[t, n_layers-1]), 6)
                                    for t in range(min(grid.shape[0], 40))],
        })
        print(f"  P{pi}: t0_p={t0_L_last:.4f} cryst_t={cryst_tok} found={answer_found}")

    math_easy_results[cond_name] = results

# Summary
print("\n=== EASY MATH SUMMARY ===")
for cond_name in conditions:
    rr = math_easy_results[cond_name]
    mean_t0 = np.mean([r["t0_p_final_layer"] for r in rr])
    cryst = [r["crystallization_token"] for r in rr if r["crystallization_token"] >= 0]
    n_found = sum(1 for r in rr if r["answer_found"])
    print(f"{cond_name}: t0_p(ans)={mean_t0:.4f}, mean_cryst={np.mean(cryst) if cryst else -1:.1f}, correct={n_found}/10")

print("\n--- 3B COMPARISON ---")
print("3B baseline_zh: t0_p=0.29, mean_cryst=4.0")
print("3B flip_zh:     t0_p=0.26, mean_cryst=0.0")
print("3B baseline_en: t0_p=0.026")

In [ ]:
# Cell 6: SECTION B — Difficulty Gradient (Easy → AIME)

DIFFICULTY_PROBLEMS = [
    # EASY
    {"prompt": "Calculate 47 + 86.", "answer": 133, "difficulty": "easy", "label": "addition"},
    {"prompt": "Calculate 15 × 8.", "answer": 120, "difficulty": "easy", "label": "multiplication"},
    {"prompt": "What is the remainder when 100 is divided by 7?", "answer": 2, "difficulty": "easy", "label": "mod"},
    # MEDIUM
    {"prompt": "What is the sum of all positive divisors of 28?", "answer": 56, "difficulty": "medium", "label": "divisor_sum"},
    {"prompt": "How many integers between 1 and 100 are divisible by 3 but not by 5?", "answer": 27, "difficulty": "medium", "label": "inclusion_exclusion"},
    {"prompt": "What is the remainder when 2^10 is divided by 7?", "answer": 2, "difficulty": "medium", "label": "modular_exp"},
    # HARD
    {"prompt": "How many 4-digit palindromes are there?", "answer": 90, "difficulty": "hard", "label": "palindromes"},
    {"prompt": "Find the last three digits of 7^2025.", "answer": 807, "difficulty": "hard", "label": "modular_power"},
    {"prompt": "Find the number of positive integer divisors of 12!.", "answer": 792, "difficulty": "hard", "label": "factorial_divisors"},
    # AIME-ADJACENT
    {"prompt": "Find the sum of all positive integers n < 1000 such that n^2 + n + 1 is divisible by both 7 and 13.", "answer": 0, "difficulty": "aime", "label": "quadratic_mod"},
    {"prompt": "Let S be the set of positive integers n such that both n and n+1 have exactly 4 positive divisors. Find the smallest element of S.", "answer": 14, "difficulty": "aime", "label": "divisor_pair"},
]

difficulty_results = []
for pi, prob in enumerate(DIFFICULTY_PROBLEMS):
    print(f"\nP{pi} [{prob['difficulty']}] {prob['label']}: {prob['prompt'][:60]}...")
    grid, gen_text, gen_toks = generate_with_crystallization(
        model, tokenizer, prob["prompt"], prob["answer"], max_new=200
    )

    t0_p_final = float(grid[0, n_layers-1]) if grid.shape[0] > 0 else 0.0

    # When does p first exceed thresholds at final layer?
    thresholds = {"0.01": 0.01, "0.05": 0.05, "0.10": 0.10, "0.50": 0.50}
    first_exceed = {}
    for name, thresh in thresholds.items():
        for t_idx in range(grid.shape[0]):
            if grid[t_idx, n_layers-1] > thresh:
                first_exceed[name] = t_idx
                break
        else:
            first_exceed[name] = -1

    ans_str = str(prob["answer"])
    answer_found = ans_str in gen_text if prob["answer"] > 0 else False

    difficulty_results.append({
        "pi": pi, "difficulty": prob["difficulty"], "label": prob["label"],
        "answer": prob["answer"], "t0_p_final": round(t0_p_final, 6),
        "first_exceed": first_exceed, "answer_found": answer_found,
        "n_tokens": grid.shape[0], "gen_text": gen_text[:300],
        "token_profile_final": [round(float(grid[t, n_layers-1]), 6)
                                for t in range(min(grid.shape[0], 50))],
    })
    print(f"  t0_p={t0_p_final:.4f} first>0.01@t={first_exceed.get('0.01', -1)} "
          f"first>0.50@t={first_exceed.get('0.50', -1)} found={answer_found}")
    print(f"  Generated: {gen_text[:120]}...")

# Summary by difficulty
print("\n=== DIFFICULTY GRADIENT ===")
for diff in ["easy", "medium", "hard", "aime"]:
    subset = [r for r in difficulty_results if r["difficulty"] == diff]
    if not subset: continue
    mean_t0 = np.mean([r["t0_p_final"] for r in subset])
    exceed_50 = [r["first_exceed"].get("0.50", -1) for r in subset
                 if r["first_exceed"].get("0.50", -1) >= 0]
    n_found = sum(1 for r in subset if r["answer_found"])
    print(f"{diff:>6}: t0_p={mean_t0:.4f}, mean_token_p>0.5={np.mean(exceed_50) if exceed_50 else -1:.1f}, correct={n_found}/{len(subset)}")

print("\n--- 3B COMPARISON ---")
print("3B: easy t0=5%, p>0.5 @ t=4.5 | medium t0=2%, @t=7.3 | hard t0=8.6%, @t=7.3 | aime t0=2.6%, @t=15.5")

In [ ]:
# Cell 7: SECTION C — Non-Math Crystallization (THE universality test)
# Does "knows before speaks" hold for factual recall, translation, reasoning?

NONMATH_PROBLEMS = [
    # FACTUAL RECALL — deterministic, single-token-ish answer
    {"prompt": "What is the capital of France?", "answer": "Paris", "category": "factual", "label": "capital_france"},
    {"prompt": "What is the chemical symbol for gold?", "answer": "Au", "category": "factual", "label": "gold_symbol"},
    {"prompt": "What planet is closest to the Sun?", "answer": "Mercury", "category": "factual", "label": "closest_planet"},
    {"prompt": "In what year did World War II end?", "answer": "1945", "category": "factual", "label": "ww2_end"},
    {"prompt": "What is the largest ocean on Earth?", "answer": "Pacific", "category": "factual", "label": "largest_ocean"},
    # TRANSLATION — does the model "know" the translation at t=0?
    {"prompt": "Translate to English: 太阳", "answer": "sun", "category": "translation", "label": "sun_zh_en"},
    {"prompt": "Translate to English: 数学", "answer": "math", "category": "translation", "label": "math_zh_en"},
    {"prompt": "Translate to Chinese: computer", "answer": "电脑", "category": "translation", "label": "computer_en_zh"},
    # LOGICAL REASONING — simple, deterministic
    {"prompt": "If all roses are flowers and all flowers need water, do roses need water? Answer yes or no.", "answer": "yes", "category": "logic", "label": "syllogism"},
    {"prompt": "What comes next in the pattern: 2, 4, 8, 16, ?", "answer": "32", "category": "logic", "label": "geometric_seq"},
    {"prompt": "If today is Wednesday, what day is it 3 days from now?", "answer": "Saturday", "category": "logic", "label": "day_calc"},
]

def get_answer_token_ids_text(tokenizer, answer_str):
    """Like get_answer_token_ids but for text answers."""
    ids = set()
    for prefix in ["", " ", "\n"]:
        for variant in [answer_str, answer_str.lower(), answer_str.upper(), answer_str.capitalize()]:
            toks = tokenizer.encode(prefix + variant, add_special_tokens=False)
            ids.update(toks)
    return list(ids)


nonmath_results = []
for pi, prob in enumerate(NONMATH_PROBLEMS):
    print(f"\nP{pi} [{prob['category']}] {prob['label']}: {prob['prompt']}")

    answer_str = str(prob["answer"])
    answer_token_ids = get_answer_token_ids_text(tokenizer, answer_str)

    # Run generation, manually compute p(answer)
    device = next(model.parameters()).device
    final_ln = model.model.norm
    lm_head_fn = model.lm_head

    input_ids = tokenizer.encode(prob["prompt"], add_special_tokens=True)
    input_ids_t = torch.tensor([input_ids], device=device)

    crystallization = []
    generated_tokens = []

    for t in range(60):
        layer_hiddens = {}
        def make_capture(l):
            def hook(module, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                layer_hiddens[l] = h[:, -1:, :].detach()
            return hook

        cap_handles = [model.model.layers[l].register_forward_hook(make_capture(l))
                       for l in ALL_LAYERS]
        with torch.no_grad():
            outputs = model(input_ids_t)
        for h in cap_handles:
            h.remove()

        next_token = outputs.logits[:, -1, :].argmax(dim=-1)

        layer_probs = {}
        for l in ALL_LAYERS:
            h_l = layer_hiddens[l]
            h_normed = final_ln(h_l)
            logits_l = lm_head_fn(h_normed).float().squeeze(0).squeeze(0)
            probs = F.softmax(logits_l, dim=-1)
            p_answer = max(probs[tid].item() for tid in answer_token_ids) if answer_token_ids else 0.0
            layer_probs[l] = p_answer

        crystallization.append(layer_probs)
        generated_tokens.append(next_token.item())
        input_ids_t = torch.cat([input_ids_t, next_token.unsqueeze(0)], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    n_tok = len(crystallization)
    grid = np.zeros((n_tok, n_layers))
    for t_idx in range(n_tok):
        for l in ALL_LAYERS:
            grid[t_idx, l] = crystallization[t_idx].get(l, 0.0)

    t0_p = float(grid[0, n_layers-1]) if grid.shape[0] > 0 else 0.0

    # Crystallization token (first t where p > 0.1 at final layer)
    cryst_tok = -1
    for t_idx in range(grid.shape[0]):
        if grid[t_idx, n_layers-1] > 0.1:
            cryst_tok = t_idx
            break

    answer_found = answer_str.lower() in gen_text.lower()

    nonmath_results.append({
        "pi": pi, "category": prob["category"], "label": prob["label"],
        "answer": answer_str, "t0_p_final": round(t0_p, 6),
        "crystallization_token": cryst_tok, "answer_found": answer_found,
        "gen_text": gen_text[:200],
        "layer_profile_t0": [round(float(grid[0, l]), 6) for l in range(n_layers)] if grid.shape[0] > 0 else [],
        "token_profile_final": [round(float(grid[t, n_layers-1]), 6) for t in range(min(n_tok, 30))],
    })
    print(f"  t0_p(ans)={t0_p:.4f} cryst_t={cryst_tok} found={answer_found}")
    print(f"  Generated: {gen_text[:100]}")

# Summary by category
print("\n=== NON-MATH SUMMARY ===")
for cat in ["factual", "translation", "logic"]:
    subset = [r for r in nonmath_results if r["category"] == cat]
    if not subset: continue
    mean_t0 = np.mean([r["t0_p_final"] for r in subset])
    cryst = [r["crystallization_token"] for r in subset if r["crystallization_token"] >= 0]
    n_found = sum(1 for r in subset if r["answer_found"])
    print(f"{cat:>12}: t0_p={mean_t0:.4f}, mean_cryst={np.mean(cryst) if cryst else -1:.1f}, correct={n_found}/{len(subset)}")

In [ ]:
# Cell 8: SECTION D — Constrained Decoding Test
# Force the model to output the answer at t=0 (no CoT).
# If accuracy matches full generation for easy problems → CoT is expression, not computation.

def constrained_decode_accuracy(model, tokenizer, problems, n_layers):
    """
    For each problem, check if the model's argmax at t=0 (final layer)
    matches any answer token. Compare with full-generation accuracy.
    """
    device = next(model.parameters()).device
    results = []

    for pi, prob in enumerate(problems):
        prompt = prob.get("en", prob.get("prompt", ""))
        answer = prob["answer"]
        answer_token_ids = get_answer_token_ids(tokenizer, answer)

        # Single forward pass — get logits at last prompt token
        input_ids = tokenizer.encode(prompt, add_special_tokens=True)
        input_ids_t = torch.tensor([input_ids], device=device)

        with torch.no_grad():
            outputs = model(input_ids_t)

        logits = outputs.logits[:, -1, :].float().squeeze(0)
        probs = F.softmax(logits, dim=-1)

        # Argmax token
        top1_id = logits.argmax().item()
        top1_str = tokenizer.decode([top1_id])

        # p(answer) and rank of answer
        p_ans = max(probs[tid].item() for tid in answer_token_ids) if answer_token_ids else 0.0
        # Rank: how many tokens have higher probability?
        best_ans_id = max(answer_token_ids, key=lambda tid: probs[tid].item()) if answer_token_ids else -1
        rank = (probs > probs[best_ans_id]).sum().item() if best_ans_id >= 0 else -1

        # Would constrained decoding get it right?
        constrained_correct = top1_id in answer_token_ids

        # Top-5 for inspection
        top5_ids = logits.topk(5).indices.tolist()
        top5 = [(tokenizer.decode([tid]), round(probs[tid].item(), 4)) for tid in top5_ids]

        results.append({
            "pi": pi, "answer": answer, "p_answer": round(p_ans, 6),
            "rank": rank, "constrained_correct": constrained_correct,
            "top1": top1_str, "top5": top5,
            "difficulty": prob.get("difficulty", "easy"),
        })

        status = "MATCH" if constrained_correct else f"MISS (top1={top1_str!r}, rank={rank})"
        print(f"  P{pi}: {status} p(ans)={p_ans:.4f} top5={top5}")

    return results


# Test on easy math (EN)
print("=== CONSTRAINED DECODING: Easy Math (EN) ===")
easy_constrained = constrained_decode_accuracy(model, tokenizer, MATH_EASY, n_layers)
n_match = sum(1 for r in easy_constrained if r["constrained_correct"])
print(f"\nConstrained accuracy (easy, EN): {n_match}/{len(MATH_EASY)}")

# Test on easy math (ZH)
print("\n=== CONSTRAINED DECODING: Easy Math (ZH) ===")
zh_problems = [{"en": p["zh"], "answer": p["answer"]} for p in MATH_EASY]  # hack: put zh in "en" field
zh_constrained = constrained_decode_accuracy(model, tokenizer, zh_problems, n_layers)
n_match_zh = sum(1 for r in zh_constrained if r["constrained_correct"])
print(f"\nConstrained accuracy (easy, ZH): {n_match_zh}/{len(MATH_EASY)}")

# Test on difficulty gradient
print("\n=== CONSTRAINED DECODING: Difficulty Gradient ===")
diff_problems = [{"en": p["prompt"], "answer": p["answer"], "difficulty": p["difficulty"]}
                 for p in DIFFICULTY_PROBLEMS if p["answer"] > 0]
diff_constrained = constrained_decode_accuracy(model, tokenizer, diff_problems, n_layers)

for diff in ["easy", "medium", "hard", "aime"]:
    subset = [r for r in diff_constrained if r["difficulty"] == diff]
    if not subset: continue
    n_c = sum(1 for r in subset if r["constrained_correct"])
    mean_p = np.mean([r["p_answer"] for r in subset])
    mean_rank = np.mean([r["rank"] for r in subset])
    print(f"{diff:>6}: constrained={n_c}/{len(subset)}, mean_p={mean_p:.4f}, mean_rank={mean_rank:.0f}")

In [ ]:
# Cell 9: SECTION E — Participation Ratio (Greg's hyperplane hypothesis)
# Does effective dimensionality narrow through the layer stack?

def participation_ratio_per_layer(model, tokenizer, prompts, n_layers):
    """
    For each layer, collect last-token hidden states across prompts.
    Compute participation ratio = (sum λ_i)^2 / sum(λ_i^2)
    This measures effective dimensionality of the representation.
    """
    device = next(model.parameters()).device
    layer_states = {l: [] for l in range(n_layers)}

    def make_hook(l):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            layer_states[l].append(h[:, -1, :].detach().cpu().float().numpy().squeeze())
        return hook

    handles = [model.model.layers[l].register_forward_hook(make_hook(l)) for l in range(n_layers)]
    try:
        for prompt in prompts:
            inp = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                model(**inp)
    finally:
        for h in handles:
            h.remove()

    pr_per_layer = []
    for l in range(n_layers):
        X = np.array(layer_states[l])  # (n_prompts, d_model)
        X = X - X.mean(axis=0)  # center
        cov = X.T @ X / X.shape[0]
        eigenvalues = np.linalg.eigvalsh(cov)
        eigenvalues = eigenvalues[eigenvalues > 0]  # positive only
        pr = (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum()
        pr_per_layer.append(float(pr))

    return pr_per_layer


# Use all math prompts (EN + ZH) for broad coverage
pr_prompts = [p["en"] for p in MATH_EASY] + [p["zh"] for p in MATH_EASY]
pr_prompts += [p["prompt"] for p in DIFFICULTY_PROBLEMS]
pr_prompts += [p["prompt"] for p in NONMATH_PROBLEMS]
print(f"Computing participation ratio across {len(pr_prompts)} prompts...")

pr_values = participation_ratio_per_layer(model, tokenizer, pr_prompts, n_layers)

print("\n=== PARTICIPATION RATIO BY LAYER ===")
print(f"{'Layer':>5} {'PR':>8} {'Bar'}")
for l, pr in enumerate(pr_values):
    bar = '#' * int(pr / max(pr_values) * 40)
    print(f"L{l:>3}: {pr:>8.1f} {bar}")

print(f"\nMax PR: {max(pr_values):.1f} at L{np.argmax(pr_values)}")
print(f"Min PR: {min(pr_values):.1f} at L{np.argmin(pr_values)}")
print(f"L0: {pr_values[0]:.1f}, L{n_layers//4}: {pr_values[n_layers//4]:.1f}, "
      f"L{n_layers//2}: {pr_values[n_layers//2]:.1f}, L{3*n_layers//4}: {pr_values[3*n_layers//4]:.1f}, "
      f"L{n_layers-1}: {pr_values[-1]:.1f}")

# Greg's hypothesis: PR should DECREASE through the layer stack (manifold narrows)
# My counter: residual stream accumulates, so PR should INCREASE
early_mean = np.mean(pr_values[:n_layers//4])
mid_mean = np.mean(pr_values[n_layers//4:3*n_layers//4])
late_mean = np.mean(pr_values[3*n_layers//4:])
print(f"\nEarly (L0-L{n_layers//4-1}): {early_mean:.1f}")
print(f"Middle (L{n_layers//4}-L{3*n_layers//4-1}): {mid_mean:.1f}")
print(f"Late (L{3*n_layers//4}-L{n_layers-1}): {late_mean:.1f}")
if late_mean < early_mean:
    print("→ Greg's hypothesis SUPPORTED: manifold narrows")
else:
    print("→ Counter-hypothesis: residual stream expands")

In [ ]:
# Cell 10: Save all results and download

all_output = {
    "experiment": "cross_model_crystallization_7b",
    "model": MODEL_NAME,
    "n_layers": n_layers,
    "d_model": d_model,
    "strip_layers": STRIP_LAYERS,
    "sections": {
        "A_math_easy": {
            "description": "Easy math crystallization: baseline ZH/EN + flip ZH",
            "results": math_easy_results,
        },
        "B_difficulty_gradient": {
            "description": "Difficulty gradient: easy/medium/hard/AIME (EN only)",
            "results": difficulty_results,
        },
        "C_nonmath": {
            "description": "Non-math: factual recall, translation, logic",
            "results": nonmath_results,
        },
        "D_constrained": {
            "description": "Constrained decoding: accuracy at t=0 vs full generation",
            "easy_en": easy_constrained,
            "easy_zh": zh_constrained,
            "difficulty": diff_constrained,
        },
        "E_participation_ratio": {
            "description": "Effective dimensionality per layer (participation ratio)",
            "values": pr_values,
            "n_prompts": len(pr_prompts),
        },
    },
    "comparison_3b": {
        "baseline_zh_t0_p": 0.29,
        "baseline_en_t0_p": 0.026,
        "flip_zh_mean_cryst": 0.0,
        "baseline_zh_mean_cryst": 4.0,
        "difficulty_gradient": "easy=t4.5, medium=t7.3, hard=t7.3, aime=t15.5",
    }
}

output_path = "crystallization_7b_results.json"
with open(output_path, "w") as f:
    json.dump(all_output, f, indent=2, ensure_ascii=False)

print(f"Saved to {output_path}")
print(f"File size: {Path(output_path).stat().st_size / 1024:.1f} KB")

# Download if on Colab
try:
    from google.colab import files
    files.download(output_path)
    print("Download triggered.")
except ImportError:
    print("Not on Colab — file saved locally.")

In [ ]:
# Cell 11: Summary printout (copy this into your notes)

print("="*60)
print("CROSS-MODEL CRYSTALLIZATION: Qwen2.5-7B")
print("="*60)

print("\n--- A: Easy Math Crystallization ---")
for cond_name in ["baseline_zh", "flip_zh", "baseline_en"]:
    rr = math_easy_results[cond_name]
    mean_t0 = np.mean([r["t0_p_final_layer"] for r in rr])
    cryst = [r["crystallization_token"] for r in rr if r["crystallization_token"] >= 0]
    n_found = sum(1 for r in rr if r["answer_found"])
    print(f"  {cond_name:>12}: t0_p={mean_t0:.4f} cryst={np.mean(cryst) if cryst else -1:.1f} correct={n_found}/10")

print("\n--- B: Difficulty Gradient ---")
for diff in ["easy", "medium", "hard", "aime"]:
    subset = [r for r in difficulty_results if r["difficulty"] == diff]
    if not subset: continue
    mean_t0 = np.mean([r["t0_p_final"] for r in subset])
    exceed = [r["first_exceed"].get("0.50", -1) for r in subset if r["first_exceed"].get("0.50", -1) >= 0]
    print(f"  {diff:>6}: t0_p={mean_t0:.4f} p>0.5_at={np.mean(exceed) if exceed else 'never'}")

print("\n--- C: Non-Math Crystallization ---")
for cat in ["factual", "translation", "logic"]:
    subset = [r for r in nonmath_results if r["category"] == cat]
    if not subset: continue
    mean_t0 = np.mean([r["t0_p_final"] for r in subset])
    cryst = [r["crystallization_token"] for r in subset if r["crystallization_token"] >= 0]
    print(f"  {cat:>12}: t0_p={mean_t0:.4f} cryst={np.mean(cryst) if cryst else 'never'}")

print("\n--- D: Constrained Decoding ---")
n_en = sum(1 for r in easy_constrained if r["constrained_correct"])
n_zh = sum(1 for r in zh_constrained if r["constrained_correct"])
print(f"  Easy EN: {n_en}/10 correct at t=0")
print(f"  Easy ZH: {n_zh}/10 correct at t=0")

print("\n--- E: Participation Ratio ---")
print(f"  Early: {early_mean:.1f} | Middle: {mid_mean:.1f} | Late: {late_mean:.1f}")
print(f"  Trend: {'NARROWING' if late_mean < early_mean else 'EXPANDING'}")

print("\n" + "="*60)
print("VERDICT: Does 'Transformers Know Before They Speak' replicate?")
print("="*60)